# Crum Soliton Gas Evolved by the ETDRK4 KdV Solver

This notebook combines the two existing workflows:

- `KdVSolitons.ipynb` / `KdVCrumProject`: generate reflectionless soliton-gas initial data by the Crum method.
- `PaperExamples_KdV.ipynb` / `paper_examples_kdv.jl`: evolve the initial data with the Fourier ETDRK4 KdV solver.

The experiment adds one faster, larger soliton to the gas and overlays a dashed free tracer soliton with the same observed initial center and speed. The Crum potential is sign-flipped before evolution so it matches the negative-soliton convention used by `paper_examples_kdv.jl`; the dashed curve does not interact with the gas and is only a visual reference.

In [9]:
include("paper_examples_kdv.jl")
include("KdVCrumProject/src/KdVCrumProject.jl")

using Plots
using Printf
using Random
using Statistics

gr()
mkpath("figures")

"figures"

## Helpers

The Crum transform is used only to construct the initial condition. The time evolution below is then computed by `solve_kdv_etdrk4` from `paper_examples_kdv.jl`.

In [ ]:
function crum_profile_float64(x, kappa, beta, shifts; digits=80)
    return Float64.(KdVCrumProject.crum_transform(x, kappa, beta; shifts=shifts, digits=digits))
end

function free_soliton_profile(x, t, kappa, shift)
    center = shift + 4 * kappa^2 * t
    return -2 * kappa^2 * KdVCrumProject.sech_stable(kappa * (x - center))^2
end

free_soliton_vector(x, t, kappa, shift) =
    [free_soliton_profile(xi, t, kappa, shift) for xi in x]

function cte_validate_genus0_parameters(N, lambda1, C)
    N > 0 || throw(ArgumentError("N must be positive"))
    lambda1 > 0 || throw(ArgumentError("lambda1 must be positive"))
    0 < C <= 1 || throw(ArgumentError("C must be in (0, 1]"))
    return nothing
end

function cte_genus0_dos(eta, lambda1; C=1.0)
    cte_validate_genus0_parameters(1, lambda1, C)
    0 < eta < lambda1 || return 0.0
    return C * eta / (pi * sqrt(lambda1^2 - eta^2))
end

function cte_genus0_soliton_density(lambda1, C)
    cte_validate_genus0_parameters(1, lambda1, C)
    return C * lambda1 / pi
end

function cte_genus0_phase_density(lambda1, C)
    cte_validate_genus0_parameters(1, lambda1, C)
    C == 1 && return Inf
    return cte_genus0_soliton_density(lambda1, C) / (1 - C)
end

function cte_genus0_spectral_parameters(N, lambda1)
    cte_validate_genus0_parameters(N, lambda1, 1.0)
    return [lambda1 * sqrt(1 - (1 - i / N)^2) for i in 1:N]
end

function cte_genus0_phase_interval(N, lambda1, C; center=0.0)
    cte_validate_genus0_parameters(N, lambda1, C)
    C == 1 && return (center, center)

    phase_density = cte_genus0_phase_density(lambda1, C)
    half_width = N / (2 * phase_density)
    return (center - half_width, center + half_width)
end

function cte_uniform_spatial_phases(N, lambda1, C;
        sampling=:uniform_deterministic,
        rng=Random.default_rng(),
        center=0.0,
        include_endpoints=false)
    left, right = cte_genus0_phase_interval(N, lambda1, C; center=center)
    C == 1 && return fill(center, N)

    sampling_sym = sampling isa Symbol ? sampling : Symbol(sampling)
    if sampling_sym in (:uniform_random, :random)
        return left .+ (right - left) .* rand(rng, N)
    end

    if sampling_sym in (:uniform_deterministic, :deterministic, :equal, :equally_spaced)
        N == 1 && return [(left + right) / 2]
        include_endpoints && return collect(range(left, right; length=N))

        phase_step = (right - left) / N
        return collect(range(left + phase_step / 2, right - phase_step / 2; length=N))
    end

    throw(ArgumentError("unknown phase sampling $(sampling); use :uniform_random or :uniform_deterministic"))
end

function congy_tovbis_el_genus0_gas(N, lambda1, C;
        phase_sampling=:uniform_deterministic,
        rng=Random.default_rng(),
        phase_center=0.0,
        phase_include_endpoints=false)
    cte_validate_genus0_parameters(N, lambda1, C)

    kappa = cte_genus0_spectral_parameters(N, lambda1)
    phases = cte_uniform_spatial_phases(
        N,
        lambda1,
        C;
        sampling=phase_sampling,
        rng=rng,
        center=phase_center,
        include_endpoints=phase_include_endpoints)

    soliton_density = cte_genus0_soliton_density(lambda1, C)
    phase_density = cte_genus0_phase_density(lambda1, C)
    phase_interval = cte_genus0_phase_interval(N, lambda1, C; center=phase_center)

    return (kappa=kappa,
            phases=phases,
            C=C,
            lambda1=lambda1,
            phase_sampling=phase_sampling,
            phase_center=phase_center,
            phase_include_endpoints=phase_include_endpoints,
            phase_interval=phase_interval,
            soliton_density=soliton_density,
            phase_density=phase_density,
            gas_length=N / soliton_density)
end

function congy_tovbis_el_initial_data(xgrid, gas;
        include_large_soliton=false,
        large_kappa=nothing,
        large_offset=14.0,
        digits=80,
        solver_sign_flip=true)
    kappa_gas = gas.kappa
    phases_gas = gas.phases
    beta_gas = KdVCrumProject.alternating_norming_constants(length(kappa_gas))

    if include_large_soliton
        large_kappa === nothing && throw(ArgumentError("large_kappa is required when include_large_soliton=true"))
        large_kappa > maximum(kappa_gas) || throw(ArgumentError("large_kappa must exceed the gas spectral edge"))
        large_shift = minimum(phases_gas) - large_offset
        kappa_all = vcat(kappa_gas, large_kappa)
        phases_all = vcat(phases_gas, large_shift)
    else
        large_shift = nothing
        kappa_all = copy(kappa_gas)
        phases_all = copy(phases_gas)
    end

    beta_all = KdVCrumProject.alternating_norming_constants(length(kappa_all))

    phi0_paper = crum_profile_float64(xgrid, kappa_all, beta_all, phases_all; digits=digits)
    phi_gas_paper = crum_profile_float64(xgrid, kappa_gas, beta_gas, phases_gas; digits=digits)

    # Congy-El-Roberti-Tovbis use positive KdV solitons. The ETDRK4 notebook
    # evolves the sign-reversed KdV convention, so q0 is sign-flipped by default.
    field_sign = solver_sign_flip ? -1.0 : 1.0
    q0 = field_sign .* phi0_paper
    q_gas_only = field_sign .* phi_gas_paper

    tracer_shift = include_large_soliton ? xgrid[argmin(q0)] : nothing
    crum_E1, crum_E2 = KdVCrumProject.conserved_errors(xgrid, phi0_paper, kappa_all)

    return (q0=q0,
            q_gas_only=q_gas_only,
            phi0_paper=phi0_paper,
            phi_gas_paper=phi_gas_paper,
            q0_crum_positive=phi0_paper,
            q_gas_only_crum_positive=phi_gas_paper,
            kappa_gas=kappa_gas,
            phases_gas=phases_gas,
            beta_gas=beta_gas,
            kappa_all=kappa_all,
            beta_all=beta_all,
            phases_all=phases_all,
            include_large_soliton=include_large_soliton,
            large_kappa=large_kappa,
            large_shift=large_shift,
            tracer_shift=tracer_shift,
            solver_sign_flip=solver_sign_flip,
            gas=gas,
            crum_E1=crum_E1,
            crum_E2=crum_E2)
end

function plot_soliton_gas_initial_data(xgrid, xplot, initial_data;
        title="Congy-El-Roberti-Tovbis KdV soliton gas initial data", size=(820, 420))
    q0_plot = interpolate_periodic(xgrid, initial_data.q0, xplot)
    gas_plot = interpolate_periodic(xgrid, initial_data.q_gas_only, xplot)

    initial_plot = plot(xplot, q0_plot;
        color=:blue,
        linewidth=2,
        label=initial_data.include_large_soliton ? "gas + large soliton" : "gas realization",
        xlabel="x",
        ylabel="q(x,0)",
        title=title,
        framestyle=:box,
        grid=false,
        legend=:topright,
        size=size)

    if initial_data.include_large_soliton
        tracer0_plot = free_soliton_vector(xplot, 0.0, initial_data.large_kappa, initial_data.tracer_shift)
        plot!(initial_plot, xplot, gas_plot;
            color=:gray,
            linestyle=:dot,
            linewidth=1.8,
            label="paper gas only")
        plot!(initial_plot, xplot, tracer0_plot;
            color=:red,
            linestyle=:dash,
            linewidth=2.2,
            label="free tracer")
    end

    return initial_plot
end

function local_trough_position(x, y, guess; window=6.0)
    idxs = findall(abs.(x .- guess) .<= window)
    isempty(idxs) && return (x[argmin(y)], minimum(y))
    local_idx = idxs[argmin(y[idxs])]
    return (x[local_idx], y[local_idx])
end

## Congy-El-Roberti-Tovbis soliton-gas initial data

The gas below follows Appendix A of Congy, El, Roberti, and Tovbis for a genus-0 diluted condensate. The gas eigenvalues are the DOS quantiles

`1 - sqrt(1 - eta_i^2 / lambda1^2) = i / N`, `i = 1, ..., N`,

and the spatial phases `x_i^0` are uniformly distributed on the paper's S-set

`[-N / (2*kappa_s), N / (2*kappa_s)]`, with `kappa_s = kappa(C)/(1-C)` and `kappa(C) = C*lambda1/pi` for genus 0.

`phase_sampling = :uniform_deterministic` uses equally spaced S-set samples for a reproducible finite-`N` realization. Use `:uniform_random` for a random realization. The random phases are intentionally not sorted against the ordered eigenvalues.

In [ ]:
rng = MersenneTwister(7)

N_gas = 80
lambda1_gas = 0.85
dilution_C = 0.8
crum_digits = 100

# Appendix A: :uniform_deterministic gives equally spaced S-set phases;
# :uniform_random gives a random finite-N soliton gas realization.
phase_sampling = :uniform_deterministic
phase_include_endpoints = false
phase_center = 0.0

include_large_soliton = true
large_kappa = 1.0
large_offset = 14.0
large_speed = 4 * large_kappa^2
large_amplitude = -2 * large_kappa^2

cte_gas = congy_tovbis_el_genus0_gas(
    N_gas,
    lambda1_gas,
    dilution_C;
    phase_sampling=phase_sampling,
    rng=rng,
    phase_center=phase_center,
    phase_include_endpoints=phase_include_endpoints)

kappa_gas = cte_gas.kappa
phases_gas = cte_gas.phases

@assert !include_large_soliton || large_kappa > maximum(kappa_gas)

phase_steps = phase_sampling in (:uniform_deterministic, :deterministic, :equal, :equally_spaced) && length(phases_gas) > 1 ? diff(phases_gas) : Float64[]

(N_gas=N_gas,
 lambda1_gas=lambda1_gas,
 dilution_C=dilution_C,
 phase_sampling=phase_sampling,
 phase_interval=cte_gas.phase_interval,
 phase_step_range=isempty(phase_steps) ? nothing : extrema(phase_steps),
 soliton_density=cte_gas.soliton_density,
 phase_density=cte_gas.phase_density,
 gas_length=cte_gas.gas_length,
 kappa_range=extrema(kappa_gas),
 phase_span=extrema(phases_gas),
 include_large_soliton=include_large_soliton,
 large_kappa=large_kappa,
 large_speed=large_speed,
 large_amplitude=large_amplitude,
 large_shift=include_large_soliton ? minimum(phases_gas) - large_offset : nothing)

## Numerical grid and initial plot

The Crum result is generated on the same periodic grid used by the ETDRK4 solver. After the sign flip, the notebook locates the actual large trough in the full Crum profile and uses that as the tracer's initial center. The dashed red curve is the free large soliton at `t = 0`; the solid blue curve is the full interacting initial field.

In [ ]:
evolution_xmin, evolution_xmax = -180.0, 190.0
evolution_n = 4072
evolution_dt = 5e-4
evolution_save_every = 100
evolution_frame_count = 81
evolution_times = evolution_dt * evolution_save_every .* collect(0:evolution_frame_count-1)
evolution_xgrid = collect(range(evolution_xmin, evolution_xmax; length=evolution_n + 1))[1:end-1]
evolution_xplot = collect(range(-125.0, 195.0; length=1500))

initial_data = congy_tovbis_el_initial_data(
    evolution_xgrid,
    cte_gas;
    include_large_soliton=include_large_soliton,
    large_kappa=large_kappa,
    large_offset=large_offset,
    digits=crum_digits,
    solver_sign_flip=true)

q0 = initial_data.q0
q_gas_only = initial_data.q_gas_only
phi0_paper = initial_data.phi0_paper
phi_gas_paper = initial_data.phi_gas_paper
q0_crum_positive = initial_data.q0_crum_positive
q_gas_only_crum_positive = initial_data.q_gas_only_crum_positive
kappa_all = initial_data.kappa_all
beta_all = initial_data.beta_all
phases_all = initial_data.phases_all
beta_gas = initial_data.beta_gas
large_shift = initial_data.large_shift
tracer_shift = initial_data.tracer_shift
crum_E1, crum_E2 = initial_data.crum_E1, initial_data.crum_E2
tracer_center_x = include_large_soliton ? tracer_shift .+ large_speed .* evolution_times : zeros(length(evolution_times))

initial_plot = plot_soliton_gas_initial_data(
    evolution_xgrid,
    evolution_xplot,
    initial_data;
    title=@sprintf("Congy-El-Roberti-Tovbis genus-0 gas, N=%d, C=%.2f, phases=%s", N_gas, dilution_C, phase_sampling))

@show minimum(q0) maximum(q0) phase_sampling large_shift tracer_shift crum_E1 crum_E2
initial_plot

## Evolve with the numerical KdV scheme

This is the same Fourier ETDRK4 solver used in `PaperExamples_KdV.ipynb`. The tracer field is computed analytically and is not used by the solver.

In [ ]:
evolution_solution = solve_kdv_etdrk4(q0, evolution_xgrid, evolution_times; dt=evolution_dt)

evolution_wavefield = hcat(
    [interpolate_periodic(evolution_xgrid, evolution_solution[t], evolution_xplot)
     for t in evolution_times]...
)

evolution_tracer_wavefield = hcat(
    [free_soliton_vector(evolution_xplot, t, large_kappa, tracer_shift)
     for t in evolution_times]...
)

(size=size(evolution_wavefield),
 tspan=(first(evolution_times), last(evolution_times)),
 large_center_span=(first(tracer_center_x), last(tracer_center_x)))

## Animation: interacting field versus free tracer

In [ ]:
evolution_ylims = extrema(vcat(vec(evolution_wavefield), vec(evolution_tracer_wavefield)))
evolution_pad = 0.06 * max(evolution_ylims[2] - evolution_ylims[1], eps(Float64))

evolution_anim = @animate for (j, t) in enumerate(evolution_times)
    frame = plot(evolution_xplot, evolution_wavefield[:, j];
        color=:blue,
        linewidth=2,
        label="interacting field",
        legend=:topright,
        xlabel="x",
        ylabel="q(x,t)",
        xlims=(first(evolution_xplot), last(evolution_xplot)),
        ylims=(evolution_ylims[1] - evolution_pad, evolution_ylims[2] + evolution_pad),
        title=@sprintf("Crum gas interaction, t = %.2f", t),
        framestyle=:box,
        grid=false,
        size=(820, 440))
    plot!(frame, evolution_xplot, evolution_tracer_wavefield[:, j];
        color=:red,
        linestyle=:dash,
        linewidth=2.2,
        label="free tracer")
    frame
end

animation_path = joinpath("figures", "crum_soliton_gas_large_interaction.gif")
animation_gif = gif(evolution_anim, animation_path; fps=20)
display(animation_gif)
animation_path

## Heatmap with the free tracer center line

The heatmap shows the interacting numerical field. The dashed red curve is the center of the noninteracting tracer: `x(t) = tracer_shift + 4*kappa^2*t`.

In [ ]:
wave_heatmap = heatmap(evolution_xplot, evolution_times, permutedims(evolution_wavefield);
    xlabel="x",
    ylabel="t",
    colorbar_title="q(x,t)",
    title="Crum soliton gas evolved by ETDRK4",
    color=:turbo,
    framestyle=:box,
    aspect_ratio=:auto,
    legend=:topright,
    size=(820, 560))

plot!(wave_heatmap, tracer_center_x, evolution_times;
    color=:red,
    linestyle=:dash,
    linewidth=2.5,
    label="free tracer center")

heatmap_path = joinpath("figures", "crum_soliton_gas_large_interaction_heatmap.png")
savefig(wave_heatmap, heatmap_path)
display(wave_heatmap)
heatmap_path

## Optional trough tracking

This crude diagnostic tracks the most negative local trough near the free tracer center. It is useful for seeing the delay or advance caused by interaction with the gas.

In [ ]:
tracked = [local_trough_position(evolution_xplot, evolution_wavefield[:, j], tracer_center_x[j]; window=8.0)
           for j in eachindex(evolution_times)]
tracked_center_x = first.(tracked)

track_plot = plot(evolution_times, tracked_center_x;
    color=:blue,
    linewidth=2,
    label="interacting local trough",
    xlabel="t",
    ylabel="x",
    title="Large-soliton trough trajectory proxy",
    framestyle=:box,
    grid=false,
    legend=:topleft,
    size=(760, 380))
plot!(track_plot, evolution_times, tracer_center_x;
    color=:red,
    linestyle=:dash,
    linewidth=2.2,
    label="free tracer center")
track_plot